In [1]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

# 1. Inicializa o motor do Apache Spark
spark = SparkSession.builder \
    .appName("Ingestao_Bacen_Inflacao") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Inicializado com Sucesso.")

# 2. Conexão com a API de Dados Abertos do Banco Central (Série 433 - IPCA)
url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
print("Extraindo dados do servidor estatal...")
response = requests.get(url_ipca)
dados_json = response.json()

# 3. Transformação Inicial (Parsing do JSON via Pandas para o Spark)
pdf = pd.DataFrame(dados_json)
df_raw = spark.createDataFrame(pdf)

# 4. Tipagem Relacional e Limpeza
df_silver = df_raw.withColumn("data", to_date(col("data"), "dd/MM/yyyy")) \
                  .withColumn("valor", col("valor").cast("double")) \
                  .withColumnRenamed("valor", "ipca_mensal")

# 5. Exibe os dados estruturados mais recentes
print("Pipeline Executado. Amostra dos Dados (Falsa Prosperidade):")
df_silver.orderBy(col("data").desc()).show(10)

# 6. Salva o dado bruto no nosso repositório em formato otimizado (Parquet)
df_silver.write.mode("overwrite").parquet("ipca_historico.parquet")

Spark Inicializado com Sucesso.
Extraindo dados do servidor estatal...
Pipeline Executado. Amostra dos Dados (Falsa Prosperidade):
+----------+-----------+
|      data|ipca_mensal|
+----------+-----------+
|2026-07-01|       0.07|
|2026-06-01|       0.16|
|2026-05-01|       0.58|
|2026-04-01|       0.67|
|2026-03-01|       0.88|
|2026-02-01|        0.7|
|2026-01-01|       0.33|
|2025-12-01|       0.33|
|2025-11-01|       0.18|
|2025-10-01|       0.09|
+----------+-----------+
only showing top 10 rows

